**`ingest_buildings`**

Script examples to import building datasets

# Configure

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import argparse

from openplaces.io.ingest import ingest_recipe_data
from openplaces.recipe import get_recipe

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest buildings using a recipe')
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to process (e.g., "US-MA")',
    default=None,
    nargs='+',
)
parser.add_argument(
    '--recipe_admin_id',
    help='Administrative unit ID for the entire recipe (e.g., "US")',
    default=None,
)
parser.add_argument(
    '--entity',
    help='Entity identifier (e.g. "building-microsoft-v2")',
    default=None,
);

# Set arguments

In [ ]:
ARGS_TEST = (
    # '--recipe_admin_id US --entity building-fema-2023 '
    '--recipe_admin_id US --entity building-usace-2022 '
    # '--recipe_admin_id US --entity building-microsoft-v2 '
    '--admin_ids US-RI'
    # '--admin_ids US-CT US-FL US-MA US-NC US-VA US-WI US-TX'
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check whether parsing worked as expected
args

# Load recipe

In [ ]:
from openplaces.utils import pretty_print
recipe = get_recipe(args.recipe_admin_id, args.entity)
pretty_print(recipe)

# Ingest building data

In [ ]:
for admin_id in args.admin_ids:
    print(f'Processing {admin_id}:')
    buildings = ingest_recipe_data(recipe, admin_id, return_result=True)

# Inspect

In [ ]:
if str(recipe['entity'].source) == 'usace':
    from openplaces.api import get_admin1, get_admin2
    admin1 = get_admin1(admin_id, geom=True)
    admin2 = get_admin2(admin_id, geom=True)
    ax = buildings.sample(frac=0.03).plot(
        'foundation_type',
        markersize=0.1,
        legend=True,
        legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
        figsize=(10, 10),
    )
    admin1.boundary.plot(ax=ax, color='black', linewidth=0.3)
    admin2.boundary.plot(ax=ax, color='black', linewidth=0.1)
    ax.axis('off')